# Tutorial 3: coupling Xsuite and WarpX

Xsuite applies a linear map for one FCC-ee superperiod (one quarter of the ring). At its end, the two bunches are converted to a common laboratory frame, written to openPMD, collided in WarpX, and converted back to Xsuite coordinates. Repeating this map reveals the coherent beam--beam $\sigma$ and $\pi$ modes.

This notebook also inspects the handoff itself. That distinction is important when beamstrahlung is enabled: an abrupt change across the WarpX call suggests a coupling problem, while a subsequent exchange between $\sigma_z$ and $\sigma_\delta$ can be ordinary synchrotron phase-space rotation.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tutorial_003_config as config
import tutorial_003_utils as utils

plt.rcParams.update({'font.size': 13, 'figure.figsize': (11, 7)})
beam = config.get_beam_parameters()

## Beam and map parameters

The teaching model uses round Gaussian beams. At an IP with $\alpha=0$,

$$\sigma_x=\sqrt{\epsilon_x\beta_x},\qquad \sigma_{p_x}=\sqrt{\epsilon_x/\beta_x},$$

and likewise in $y$. The full-ring tunes are divided by four because one saved sample corresponds to one superperiod and one collision, not one complete revolution.

In [ ]:
for key in ('p0c_eV', 'sigma_x', 'sigma_y', 'sigma_z', 'sigma_delta', 'qx', 'qy', 'qs'):
    print(f'{key:>14s} = {beam[key]:.8g}')

## Load the available runs

The driver chooses distinct directories for the arc-only reference, the classical WarpX collision, and the collision with beamstrahlung. Missing directories are skipped, so you can analyze whichever runs you completed.

In [ ]:
candidates = {
    'arc only': Path('outputs_without_warpx'),
    'WarpX': Path('outputs_with_warpx'),
    'WarpX + beamstrahlung': Path('outputs_with_warpx_beamstrahlung'),
}
runs = {name: utils.read_moment_csv(path) for name, path in candidates.items()
        if (path / 'moments_b1.csv').exists()}
if not runs:
    raise FileNotFoundError('Run at least one simulation before executing this notebook.')
print('Loaded:', ', '.join(runs))

## Centroids, rms sizes, and longitudinal emittance

The rms sizes show envelope motion, while the centroids reveal coherent dipole motion. The longitudinal rms emittance is calculated from the covariance matrix,

$$\epsilon_\zeta=\sqrt{\sigma_\zeta^2\sigma_\delta^2-\operatorname{Cov}(\zeta,\delta)^2}. $$

A beamstrahlung kick primarily changes $\delta$. The following Xsuite map rotates the resulting mismatch in $(\zeta,\delta)$ phase space, so $\sigma_z$ can oscillate even when the handoff is exact.

In [ ]:
normalizers = {
    'x_std': beam['sigma_x'], 'y_std': beam['sigma_y'],
    'zeta_std': beam['sigma_z'], 'delta_std': beam['sigma_delta'],
}
fig, axes = plt.subplots(3, 2, figsize=(13, 11), sharex=True)
for run_name, data in runs.items():
    for beam_number, linestyle in ((1, '-'), (2, '--')):
        d = data[beam_number]
        label = f'{run_name}, beam {beam_number}'
        axes[0, 0].plot(d['iteration'], d['x_std']/beam['sigma_x'], linestyle, label=label)
        axes[0, 1].plot(d['iteration'], d['y_std']/beam['sigma_y'], linestyle, label=label)
        axes[1, 0].plot(d['iteration'], d['zeta_std']/beam['sigma_z'], linestyle, label=label)
        axes[1, 1].plot(d['iteration'], d['delta_std']/beam['sigma_delta'], linestyle, label=label)
        axes[2, 0].plot(d['iteration'], d['emit_zeta']/d['emit_zeta'][0], linestyle, label=label)
        axes[2, 1].plot(d['iteration'], d['delta_mean']/beam['sigma_delta'], linestyle, label=label)
labels = (r'$\sigma_x/\sigma_{x0}$', r'$\sigma_y/\sigma_{y0}$',
          r'$\sigma_\zeta/\sigma_{\zeta0}$', r'$\sigma_\delta/\sigma_{\delta0}$',
          r'$\epsilon_\zeta/\epsilon_{\zeta,1}$', r'$\langle\delta\rangle/\sigma_{\delta0}$')
for ax, ylabel in zip(axes.flat, labels):
    ax.set_ylabel(ylabel); ax.grid(alpha=.3)
for ax in axes[-1]: ax.set_xlabel('superperiod iteration')
axes[0, 0].legend(fontsize=8, ncol=2)
fig.tight_layout()

## Diagnose the handoff

`handoff_checks.csv` compares each bunch immediately before and after WarpX. With beamstrahlung disabled, the momentum round-trip error should be near floating-point precision and no primary particles should disappear. The adapter also converts between Xsuite's IP-plane coordinates and simultaneous WarpX snapshots with each particle's own longitudinal velocity. With beamstrahlung enabled, $\langle\Delta\delta\rangle<0$ is expected, but a large instantaneous $\zeta$ jump is suspicious because ultra-relativistic particles remain very close to $c$ during this short collision.

In [ ]:
handoffs = {}
for name, path in candidates.items():
    check_file = path / 'handoff_checks.csv'
    if check_file.exists(): handoffs[name] = pd.read_csv(check_file)

if handoffs:
    fig, axes = plt.subplots(2, 3, figsize=(16, 8), sharex=True)
    for name, table in handoffs.items():
        for beam_number, linestyle in ((1, '-'), (2, '--')):
            d = table[table.beam == beam_number]
            label = f'{name}, beam {beam_number}'
            axes[0, 0].plot(d.iteration, d.mean_delta_change, linestyle, label=label)
            axes[0, 1].plot(d.iteration, d.rms_zeta_change_m, linestyle, label=label)
            axes[0, 2].plot(d.iteration, d.emit_zeta_after_m/d.emit_zeta_before_m, linestyle, label=label)
            axes[1, 0].plot(d.iteration, d.input_momentum_roundtrip_max, linestyle, label=label)
            axes[1, 1].plot(d.iteration, d.input_position_roundtrip_max_m, linestyle, label=label)
            axes[1, 2].plot(d.iteration, d.n_returned/d.n_sent, linestyle, label=label)
    axes[0, 0].set_ylabel(r'$\langle\Delta\delta\rangle$')
    axes[0, 1].set_ylabel(r'rms$(\Delta\zeta)$ [m]')
    axes[0, 2].set_ylabel(r'$\epsilon_{\zeta,after}/\epsilon_{\zeta,before}$')
    axes[1, 0].set_ylabel('momentum round-trip error')
    axes[1, 1].set_ylabel('position round-trip error [m]')
    axes[1, 2].set_ylabel('returned / sent primaries')
    for ax in axes.flat: ax.grid(alpha=.3)
    for ax in axes[-1]: ax.set_xlabel('superperiod iteration')
    axes[0, 0].legend(fontsize=8)
    fig.tight_layout()
else:
    print('No WarpX handoff log is available.')

## Coherent $\sigma$ and $\pi$ modes

For equal counter-rotating beams, the in-phase physical motion is the $\sigma$ mode and remains near the unperturbed tune. The out-of-phase motion is the $\pi$ mode and receives the coherent beam--beam tune shift. Because the local horizontal axes of the two beams point oppositely,

$$x_\sigma=x_1-x_2,\qquad x_\pi=x_1+x_2,$$

whereas the shared vertical orientation gives $y_\sigma=y_1+y_2$ and $y_\pi=y_1-y_2$. A real sampled signal cannot distinguish $Q$ from $1-Q$; here the spectrum is displayed on the $Q>0.5$ branch. Its bin spacing is $\Delta Q=1/N$, so the 20-step laptop run is only a workflow check.

In [ ]:
def spectrum_on_upper_branch(signal):
    signal = np.asarray(signal)
    windowed = (signal - np.mean(signal)) * np.hanning(len(signal))
    amplitude = np.abs(np.fft.rfft(windowed))
    amplitude /= max(np.max(amplitude), np.finfo(float).tiny)
    db = 20*np.log10(np.maximum(amplitude, np.finfo(float).tiny))
    tune = 1.0 - np.fft.rfftfreq(len(signal))
    return tune[::-1], db[::-1]

fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
for run_name, data in runs.items():
    x1, x2 = data[1]['x_mean'], data[2]['x_mean']
    y1, y2 = data[1]['y_mean'], data[2]['y_mean']
    for signal, mode, ax in ((x1-x2, r'$x_\sigma$', axes[0]),
                             (x1+x2, r'$x_\pi$', axes[0]),
                             (y1+y2, r'$y_\sigma$', axes[1]),
                             (y1-y2, r'$y_\pi$', axes[1])):
        q, amplitude = spectrum_on_upper_branch(signal)
        ax.plot(q, amplitude, label=f'{run_name} {mode}')
for ax, plane in zip(axes, ('horizontal', 'vertical')):
    ax.set_xlim(.5, .75); ax.set_ylim(-100, 3); ax.grid(alpha=.3)
    ax.set_xlabel('superperiod tune $Q$'); ax.set_title(plane); ax.legend(fontsize=8)
axes[0].set_ylabel('normalized amplitude [dB]')
fig.tight_layout()

## How to interpret a beamstrahlung $\sigma_z$ oscillation

1. Check `rms_zeta_change_m`. A large jump during WarpX points to the coordinate/time handoff.
2. Check `mean_delta_change` and the before/after energy spread. These should respond directly to photon emission.
3. Check the next arc step. If $\sigma_\delta$ changes at the collision and $\sigma_z$ changes mainly during later Xsuite transport, the bunch has become mismatched and the longitudinal map is rotating that mismatch.
4. Check $\epsilon_\zeta$. Pure linear synchrotron rotation preserves it; stochastic emission can increase it.
5. Compare several random seeds. The driver deliberately uses a different reproducible WarpX seed at every collision so that restarting WarpX does not repeat the same Monte Carlo sequence each superperiod. It also leaves `opticalDepthQSR` for WarpX to initialize independently. The zero-valued record in the earlier handoff was probably ignored or overwritten by current WarpX, so it is not by itself evidence for the observed oscillation.